<a href="https://colab.research.google.com/github/Kalrfou/Special_Topics2/blob/main/Example_chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Simple Messages Structure Demo

This chatbot demonstrates the basic structure used by modern Large Language Models (LLMs).

The main goal is to help students understand:
- `messages`
- `role`
- `system`
- `user`
- `content`

The chatbot does not use a real AI model yet. Instead, it shows how conversations are formatted before being sent to an LLM API.

## Key Concepts
- The `system` role controls the behavior of the assistant.
- The `user` role contains the user request.
- Messages are stored as structured conversation objects.

This is the foundation of:
- ChatGPT
- AI assistants
- RAG systems
- Modern LLM APIs

---

In [ ]:
!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline

In [ ]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

# 2. Stateless Phi-3 Mini Chatbot

This chatbot uses the open-source model:

```python
microsoft/Phi-3-mini-4k-instruct
```

The chatbot is **stateless**, meaning:
- it does NOT remember previous messages,
- every question is treated independently.

This type of chatbot is useful when:
- each request should be isolated,
- memory is not required,
- we want simple inference behavior.

## Key Concepts
- Hugging Face Transformers
- Pipelines API
- Prompt templates
- System messages
- Inference with open-source LLMs

## Advantages
- Simple
- Fast
- Lower memory usage

## Disadvantage
- No conversation memory.

---

In [ ]:
import gradio as gr
from transformers import pipeline
import torch

# ---------------------------------------------------
# Load Phi model
# ---------------------------------------------------
pipe = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct",
    device_map="auto",
    torch_dtype=torch.float16
)

# ---------------------------------------------------
# System Message
# ---------------------------------------------------
SYSTEM_MESSAGE = """
You are a helpful AI assistant.
Answer clearly and briefly.
"""

# ---------------------------------------------------
# Document
# ---------------------------------------------------
document = """
Large Language Models use messages with roles such as:
- system
- user
- assistant

The system role controls behavior.
The user role contains the request.
"""

# ---------------------------------------------------
# Prompt Template
# ---------------------------------------------------
prompt_template = """
Use the following document to answer the question.

DOCUMENT:
[DOCUMENT]

QUESTION:
[QUESTION]

ANSWER:
"""

# ---------------------------------------------------
# Chat Function
# ---------------------------------------------------
def chatbot(message, history):

    # Replace placeholders
    final_prompt = prompt_template.replace(
        "[DOCUMENT]", document
    ).replace(
        "[QUESTION]", message
    )

    # Create messages format
    messages = [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE
        },
        {
            "role": "user",
            "content": final_prompt
        }
    ]

    # Generate response
    output = pipe(
        messages,
        max_new_tokens=70,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    # Extract generated text
    response = output[0]["generated_text"][-1]["content"]

    return response

# ---------------------------------------------------
# Gradio UI
# ---------------------------------------------------
demo = gr.ChatInterface(
    fn=chatbot,
    type="messages",
    title="Phi-3 Mini Chatbot",
    description="Simple chatbot using Phi-3-mini with Gradio"
)

demo.launch()

# 3. Phi-3 Mini Chatbot with Memory

This chatbot extends the previous version by adding:
- conversation history,
- multi-turn interaction,
- contextual memory.

The chatbot stores previous user and assistant messages and sends them again to the model during each new request.

This allows the model to:
- remember previous questions,
- maintain conversation context,
- generate more natural dialogue.

## Key Concepts
- Conversation memory
- Multi-turn chat
- History management
- Message accumulation

## Advantages
- More realistic chatbot behavior
- Better contextual understanding

## Disadvantages
- Higher memory usage
- Longer prompts
- Slower generation over time

In [ ]:
import torch
import gradio as gr
from transformers import pipeline

# Load the open-source LLM from Hugging Face
'''
chatbot = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct" # HuggingFaceTB/SmolLM2-360M-Instruct", # try this one as well!
    torch_dtype=torch.float16# torch.float32, # float32 use when we run the model in CPU
    device="auto"#-1 # -1 means that we run on CPU
)
'''
# System instruction
system_message = """
You are a helpful AI chatbot.
Answer clearly and simply.
If the user asks about programming, explain step by step.
"""

def respond(message, history):
    # Build conversation text
    conversation = system_message + "\n"

    for user_msg, bot_msg in history:
        conversation += "User: " + user_msg + "\n"
        conversation += "Assistant: " + bot_msg + "\n"

    conversation += "User: " + message + "\n"
    conversation += "Assistant: "

    # Generate response
    output = chatbot(
        conversation,
        max_new_tokens=75,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    # Extract only new answer
    full_text = output[0]["generated_text"]
    answer = full_text[len(conversation):]

    return answer.strip()


# Create ChatGPT-like web interface
demo = gr.ChatInterface(
    fn=respond,
    title="Hugging Face Open LLM Chatbot",
    description="A simple chatbot using an open-source LLM from Hugging Face."
)

demo.launch()

In [ ]:
!pip install -q huggingface_hub transformers accelerate

In [ ]:
!huggingface-cli whoami

# 4. Llama Chatbot with Memory

This chatbot uses Meta’s open-source Llama model:

```python
meta-llama/Llama-3.2-1B-Instruct
```

The chatbot supports:
- full conversation memory,
- multi-turn interaction,
- instruction-following behavior.

This example demonstrates how modern open-source LLMs can be used locally or in Google Colab using Hugging Face Transformers.

## Key Concepts
- Open-source LLMs
- Hugging Face Hub
- Token authentication
- Chat templates
- Context management

## Important Notes
- Some Llama models require Hugging Face authentication.
- GPU acceleration is highly recommended.
- Conversation history increases context size.

## Advantages
- Strong instruction-following
- Better conversational quality
- Open-source flexibility

## Disadvantages
- Requires more compute resources
- May require Hugging Face access approval

In [ ]:
import gradio as gr
from transformers import pipeline
import torch

pipe = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.1-8B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16
)

SYSTEM_MESSAGE = "You are a helpful AI assistant. Answer clearly."

def chatbot(message, history):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE}
    ]

    # Remember full conversation
    for chat in history:
        messages.append(chat)

    # Add current user message
    messages.append({
        "role": "user",
        "content": message
    })

    output = pipe(
        messages,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    response = output[0]["generated_text"][-1]["content"]
    return response

demo = gr.ChatInterface(
    fn=chatbot,
    type="messages",
    title="Llama Chatbot with Memory",
    description="This chatbot remembers the full conversation history."
)

demo.launch()

# 5. Prompt Template + Document Injection Chatbot

This chatbot demonstrates:
- prompt engineering,
- document injection,
- template-based prompting.

The chatbot dynamically inserts a document into a prompt using:

```python
prompt.replace("[DOCUMENT]", document)
```

This technique is commonly used in:
- RAG systems,
- summarization systems,
- document question answering,
- AI assistants.

## Key Concepts
- Prompt templates
- Dynamic prompting
- Context injection
- Retrieval-Augmented Generation (RAG)

## Advantages
- Reusable prompts
- Flexible document processing
- Structured prompting

## Disadvantages
- Large documents increase token usage
- Prompt engineering quality affects output quality

---

# 6. Gradio Chat Interface

All chatbot examples use:

```python
gradio.ChatInterface
```

Gradio provides an easy way to build:
- chatbot interfaces,
- AI demos,
- interactive web applications.

## Key Features
- Browser-based UI
- Easy deployment
- Supports chat history
- Supports message formatting

## Advantages
- Very beginner-friendly
- Fast prototyping
- Works well with Hugging Face models

## Common Usage
- AI demos
- Research prototypes
- Educational projects
- LLM experimentation